In [1]:
import pandas as pd
import numpy as np

# ===================== 【核心设置】你只改这里 =====================
delta_t_target = 10    # 你想要的目标步长（比如 10）
multiscale = True     # True=多尺度拼接(1~10) | False=单尺度(只算10)
input_file = "20260323_Matched.csv"  # 可以是 csv / xlsx / xls
# =================================================================

# 定义差分函数（不变）
def compute_difference(data, delta_t=1, relative=False, eps=1e-6):
    if delta_t < 1 or delta_t >= len(data):
        raise ValueError("delta_t must be >= 1 and < number of samples")
    x_future = data[delta_t:]
    x_now = data[:-delta_t]
    if relative:
        diff = (x_future - x_now) / (x_now + eps)
    else:
        diff = x_future - x_now
    return diff

from tqdm.auto import tqdm

# ===================== 自适应读取文件 =====================
print(f"正在读取文件: {input_file}")
if input_file.endswith(".csv"):
    df = pd.read_csv(input_file)
elif input_file.endswith((".xlsx", ".xls")):
    df = pd.read_excel(input_file)
else:
    raise ValueError("❌ 不支持的文件格式！仅支持 .csv / .xlsx / .xls")

# 数据清洗（保留数字列 + 填充空值）
df = df.select_dtypes(include=[np.number])
df = df.ffill().replace([np.inf, -np.inf], 0)
data = df.values

# ============== 核心逻辑 ==============
if multiscale:
    # 多尺度：计算 1 ~ delta_t_target 全部步长，并拼接
    all_diffs = []
    for dt in tqdm(range(1, delta_t_target + 1)):
        diff = compute_difference(data, delta_t=dt, relative=False)
        diff_df = pd.DataFrame(diff, columns=df.columns)
        all_diffs.append(diff_df)
    final_df = pd.concat(all_diffs, axis=0, ignore_index=True)
    filename = f"20260323_matched_derivative_multiscale_{delta_t_target}.csv"
else:
    # 单尺度：只计算 1 次 delta_t_target
    diff = compute_difference(data, delta_t=delta_t_target, relative=False)
    final_df = pd.DataFrame(diff, columns=df.columns)
    filename = f"20260323_matched_derivative_single_scale_{delta_t_target}_simplified超级精简版.csv"

# 保存
final_df.to_csv(filename, index=False, encoding="utf-8-sig")
print(f"✅ 保存成功: {filename}")

正在读取文件: 20260323_Matched.csv


  0%|          | 0/10 [00:00<?, ?it/s]

✅ 保存成功: 20260323_matched_derivative_multiscale_10.csv
